# Explorar os dados do BRPMM

Este notebook **só lê** os dados (não altera nada em `input-data`) e mostra:

1. As primeiras linhas de cada ficheiro importante
2. Quantos instantes tem cada lote
3. Um lote inteiro, sensor a sensor
4. Lotes aceites vs. rejeitados lado a lado
5. Resumos por lote (média, desvio, ...) por classe
6. Se os dados de teste se parecem com os de treino

Cores: **azul = accepted**, **laranja = rejected**.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Pasta dos dados na sessão Renku (pode mudar-se com a variável de ambiente BRPMM_DATA)
DATA = Path(os.environ.get("BRPMM_DATA", "/home/renku/work/input-data/Project 2 - BRPMM"))
assert DATA.is_dir(), f"Não encontrei a pasta {DATA}"

COR = {"accepted": "#2a78d6", "rejected": "#eb6834"}            # azul / laranja
COR_CONJUNTO = {"calibração": "#6250d6", "teste": "#1baf7a"}    # violeta / verde-água

pd.set_option("display.max_columns", 30, "display.width", 200)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "axes.titlesize": 9, "lines.linewidth": 1.0,
})


def caminho(p, parte, ficheiro):
    """caminho('A', 'calibration data', 'calibration_labels.csv')"""
    return DATA / f"process {p}" / parte / ficheiro


def sensores_de(df):
    return [c for c in df.columns if c not in ("batch_id", "t", "label")]


def grelha(n, ncol=4, altura=2.2):
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(14, altura * nrow), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


def legenda(fig, cores):
    handles = [Line2D([], [], color=c, lw=2, label=k) for k, c in cores.items()]
    fig.legend(handles=handles, loc="upper right", frameon=False, ncol=len(cores))


print("Dados em:", DATA)
for f in sorted(DATA.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(DATA)}  ({f.stat().st_size / 1e6:.1f} MB)")

## 1. A folha de respostas: `BRPMM_template.csv`

Uma linha por lote de **teste** (161 de A + 40 de B). A coluna `label` vem toda a `rejected`:
é aqui que vão as nossas previsões.

In [ ]:
template = pd.read_csv(DATA / "BRPMM_template.csv", dtype={"batch_id": str})
print("forma:", template.shape)
display(template.head())
template.groupby(["project", "label"]).size()

## 2. As respostas de treino: `calibration_labels.csv`

Uma linha por lote de **treino**, com a resposta certa.

In [ ]:
labels = {p: pd.read_csv(caminho(p, "calibration data", "calibration_labels.csv"), dtype={"batch_id": str})
          for p in "AB"}
for p, df in labels.items():
    print(f"\n=== Processo {p}: {len(df)} lotes ===")
    display(df.head())
    print(df.label.value_counts().to_string())

## 3. As medições dos sensores: `calibration_data.csv` e `test_data.csv`

Uma linha por **instante** de cada lote. Não há coluna de tempo, por isso crio `t` = posição da
linha dentro do lote (0, 1, 2, ...). Junto também a `label` aos dados de treino.

Isto demora uns segundos (são ~270 MB).

In [ ]:
calib, teste = {}, {}
for p in "AB":
    calib[p] = pd.read_csv(caminho(p, "calibration data", "calibration_data.csv"), dtype={"batch_id": str})
    teste[p] = pd.read_csv(caminho(p, "test data", "test_data.csv"), dtype={"batch_id": str})
    for d in (calib[p], teste[p]):
        d["t"] = d.groupby("batch_id").cumcount()
    calib[p] = calib[p].merge(labels[p], on="batch_id", how="left")
    print(f"Processo {p}: treino {calib[p].shape}, {calib[p].batch_id.nunique()} lotes | "
          f"teste {teste[p].shape}, {teste[p].batch_id.nunique()} lotes | "
          f"{len(sensores_de(calib[p]))} sensores")

In [ ]:
print("Processo A, treino:")
display(calib["A"].head())
print("Processo A, teste:")
display(teste["A"].head())

In [ ]:
print("Processo B, treino:")
display(calib["B"].head())
print("Processo B, teste:")
display(teste["B"].head())

Estatísticas de cada sensor (todas as linhas de todos os lotes de treino juntas).
Repara em sensores constantes (`std` = 0) — não trazem informação.

In [ ]:
for p in "AB":
    print(f"=== Processo {p} ===")
    display(calib[p][sensores_de(calib[p])].describe().T.round(3))

## 4. Quantos instantes tem cada lote?

Em A os lotes têm durações diferentes (alguns muito curtos). Será que os curtos são os rejeitados?

In [ ]:
duracao = {p: calib[p].groupby("batch_id").size().rename("n_instantes").to_frame()
              .join(labels[p].set_index("batch_id")) for p in "AB"}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, p in zip(axes, "AB"):
    n = duracao[p]
    ax.hist([n.loc[n.label == lab, "n_instantes"] for lab in COR], bins=30, stacked=True,
            color=list(COR.values()), label=list(COR), edgecolor="white", linewidth=0.5)
    ax.set(title=f"Processo {p}: duração dos lotes de treino", xlabel="nº de instantes", ylabel="nº de lotes")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print("Lotes mais curtos do processo A:")
display(duracao["A"].sort_values("n_instantes").head(10))
print(duracao["A"].groupby("label").n_instantes.describe().round(0))

## 5. Um lote inteiro, sensor a sensor

Muda o `batch_id` para ver outro lote. Passa `dados=teste["A"]` para ver um lote de teste.

In [ ]:
def ver_lote(p, batch_id=None, dados=None):
    dados = calib[p] if dados is None else dados
    batch_id = batch_id or dados.batch_id.iloc[0]
    d = dados[dados.batch_id == batch_id]
    lab = d["label"].iloc[0] if "label" in d else "teste (sem label)"
    sensores = sensores_de(d)
    fig, axes = grelha(len(sensores))
    for ax, s in zip(axes, sensores):
        ax.plot(d.t, d[s], color=COR.get(lab, "#52514e"))
        ax.set_title(s)
    fig.suptitle(f"Processo {p}, lote {batch_id}: {lab}", y=1.0)
    plt.tight_layout()
    plt.show()


ver_lote("A")

In [ ]:
ver_lote("B")

## 6. Aceites vs. rejeitados, sobrepostos

Para cada sensor, desenha alguns lotes aceites (azul) e alguns rejeitados (laranja).
Procura sensores onde o laranja se comporta de forma diferente do azul.

In [ ]:
def comparar(p, n_por_classe=8, seed=0):
    d = calib[p]
    rng = np.random.default_rng(seed)
    grupos = {}
    for lab in COR:
        ids = labels[p].loc[labels[p].label == lab, "batch_id"].to_numpy()
        escolhidos = rng.choice(ids, size=min(n_por_classe, len(ids)), replace=False)
        grupos[lab] = [d[d.batch_id == b] for b in escolhidos]
    sensores = sensores_de(d)
    fig, axes = grelha(len(sensores))
    for ax, s in zip(axes, sensores):
        for lab in COR:                      # rejeitados desenhados por cima
            for x in grupos[lab]:
                ax.plot(x.t, x[s], color=COR[lab], alpha=0.6, lw=0.8)
        ax.set_title(s)
    legenda(fig, COR)
    fig.suptitle(f"Processo {p}: {n_por_classe} lotes por classe", x=0.01, ha="left", y=1.0)
    plt.tight_layout()
    plt.show()


comparar("A")

In [ ]:
comparar("B", n_por_classe=5)   # B só tem 5 rejeitados

## 7. Resumo por lote, por classe

Cada lote passa a um número por sensor (a média, por defeito). Cada ponto é um lote.
Se as caixas azul e laranja estiverem separadas, esse sensor ajuda a distinguir.
Experimenta `estatistica="std"`, `"max"`, `"min"` ou `"last"`.

In [ ]:
def caixas(p, estatistica="mean"):
    d = calib[p]
    sensores = sensores_de(d)
    r = d.groupby("batch_id")[sensores].agg(estatistica).join(labels[p].set_index("batch_id"))
    rng = np.random.default_rng(0)
    fig, axes = grelha(len(sensores), altura=2.4)
    for ax, s in zip(axes, sensores):
        for i, lab in enumerate(COR):
            v = r.loc[r.label == lab, s].dropna()
            ax.boxplot([v], positions=[i], widths=0.5, showfliers=False,
                       medianprops={"color": "#0b0b0b"}, boxprops={"color": "#898781"},
                       whiskerprops={"color": "#898781"}, capprops={"color": "#898781"})
            ax.scatter(i + rng.uniform(-0.15, 0.15, len(v)), v, s=10, color=COR[lab], alpha=0.7, zorder=3)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(list(COR))
        ax.set_title(s)
    fig.suptitle(f"Processo {p}: {estatistica} de cada sensor por lote", x=0.01, ha="left", y=1.0)
    plt.tight_layout()
    plt.show()


caixas("A")

In [ ]:
caixas("B")

## 8. O teste parece-se com o treino?

Média de cada sensor por lote, treino (violeta) vs. teste (verde-água).
Se as distribuições forem muito diferentes, o que o modelo aprende no treino pode não servir no teste.

In [ ]:
def treino_vs_teste(p, estatistica="mean"):
    sensores = sensores_de(calib[p])
    a = calib[p].groupby("batch_id")[sensores].agg(estatistica)
    b = teste[p].groupby("batch_id")[sensores].agg(estatistica)
    fig, axes = grelha(len(sensores))
    for ax, s in zip(axes, sensores):
        bins = np.histogram_bin_edges(pd.concat([a[s], b[s]]).dropna(), bins=20)
        ax.hist(a[s], bins=bins, histtype="step", lw=1.5, color=COR_CONJUNTO["calibração"])
        ax.hist(b[s], bins=bins, histtype="step", lw=1.5, color=COR_CONJUNTO["teste"])
        ax.set_title(s)
    legenda(fig, COR_CONJUNTO)
    fig.suptitle(f"Processo {p}: {estatistica} por lote, treino vs. teste", x=0.01, ha="left", y=1.0)
    plt.tight_layout()
    plt.show()


treino_vs_teste("A")

In [ ]:
treino_vs_teste("B")

## O que anotar enquanto vês

- Que sensores separam aceites de rejeitados (secções 6 e 7)?
- Os lotes curtos de A são rejeitados (secção 4)?
- Há sensores constantes ou quase (secção 3) que podemos ignorar?
- O teste tem o mesmo aspeto que o treino (secção 8)?